# Phase 7: Evaluation

GitHub issue: TBD (filed after this notebook runs, with its real numbers). Depends on #7 (Phase 6:
chosen model = LightGBM reranker, saved to `data/frozen/reranker_lgbm.txt`).

**This is the single, official use of the locked test set** (639,037 examples, from `val_end` onward -
see Phase 2 notebook). Every cell below that touches `test` runs exactly once. If this notebook is ever
rerun, the test result becomes a second use and must be reported as such - it stops being a clean,
unbiased estimate.

In [1]:
from pathlib import Path
import sys

import lightgbm as lgb
import numpy as np
import polars as pl

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
FROZEN_DIR = ROOT / "data" / "frozen"

from ranking.eval.bootstrap import bootstrap_ci, paired_bootstrap_pvalue
from ranking.eval.metrics import ndcg_at_k

FEATURE_COLS = ["score", "item_popularity", "prefix_len", "category_match", "category", "query_category"]

def load(split):
    features = pl.read_parquet(FROZEN_DIR / "features" / f"{split}.parquet")
    ex = pl.read_parquet(FROZEN_DIR / "ranking_examples.parquet").filter(pl.col("split") == split)
    return (
        features.join(ex.select("session", "prefix_len"), on="session", how="left")
        .with_columns(pl.col("category_match").cast(pl.Int8))
        .sort(["session", "rank"])
    ), ex

test, test_examples = load("test")
print(f"test: {test.height:,} candidate rows, {test_examples.height:,} examples")

booster = lgb.Booster(model_file=str(FROZEN_DIR / "reranker_lgbm.txt"))
test = test.with_columns(pl.Series("pred", booster.predict(test.select(FEATURE_COLS).to_pandas())))

test: 29,961,375 candidate rows, 639,037 examples


## Step 1: Evaluate once on the locked test set

In [2]:
def per_session_ndcg(df, examples, score_col):
    ranked = df.sort(["session", score_col], descending=[False, True])
    per_session = ranked.group_by("session", maintain_order=True).agg(pl.col("grade").alias("gains"))
    joined = examples.select("session").join(per_session, on="session", how="left")
    return [ndcg_at_k(g, k=10) if g is not None else 0.0 for g in joined["gains"].to_list()]

baseline_test_ndcg = per_session_ndcg(test, test_examples, "score")
reranker_test_ndcg = per_session_ndcg(test, test_examples, "pred")

b_mean, b_lo, b_hi = bootstrap_ci(baseline_test_ndcg, seed=0)
r_mean, r_lo, r_hi = bootstrap_ci(reranker_test_ndcg, seed=0)
diff, p_value = paired_bootstrap_pvalue(reranker_test_ndcg, baseline_test_ndcg, seed=0)

print(f"TEST baseline NDCG@10: {b_mean:.4f} [{b_lo:.4f}, {b_hi:.4f}]")
print(f"TEST reranker NDCG@10: {r_mean:.4f} [{r_lo:.4f}, {r_hi:.4f}]")
print(f"paired diff: {diff:.4f}, p={p_value:.4f}")

TEST baseline NDCG@10: 0.0235 [0.0232, 0.0239]
TEST reranker NDCG@10: 0.0431 [0.0426, 0.0435]
paired diff: 0.0195, p=0.0000


## Step 2: Operating threshold

This is a ranking task, not a binary classifier, so a probability threshold doesn't directly apply.
The equivalent decision from Phase 0 is *how many slots to show* - already fixed at K=10 there
(NDCG@10, matching the business framing of a top-10 result list). No separate threshold is chosen here;
K=10 stands as decided.

## Step 3: Slice analysis

In [3]:
test_pred = test.sort(["session", "pred"], descending=[False, True]).group_by(
    "session", maintain_order=True
).agg(pl.col("grade").alias("gains"))

joined = test_examples.select("session", "prefix_len", "query_category").join(
    test_pred, on="session", how="left"
)
joined = joined.with_columns(
    pl.Series(
        "ndcg10",
        [ndcg_at_k(g, k=10) if g is not None else 0.0 for g in joined["gains"].to_list()],
    )
)

slices = joined.with_columns(
    pl.when(pl.col("prefix_len") <= 3).then(pl.lit("short (<=3)"))
    .when(pl.col("prefix_len") <= 10).then(pl.lit("medium (4-10)"))
    .otherwise(pl.lit("long (>10)")).alias("prefix_bucket"),
    (pl.col("query_category") == -1).alias("unknown_query"),
)

print("--- by prefix length ---")
print(slices.group_by("prefix_bucket").agg(pl.len().alias("n"), pl.col("ndcg10").mean().alias("mean_ndcg10")).sort("prefix_bucket"))
print()
print("--- by query-category known/unknown ---")
print(slices.group_by("unknown_query").agg(pl.len().alias("n"), pl.col("ndcg10").mean().alias("mean_ndcg10")))

--- by prefix length ---
shape: (3, 3)
┌───────────────┬────────┬─────────────┐
│ prefix_bucket ┆ n      ┆ mean_ndcg10 │
│ ---           ┆ ---    ┆ ---         │
│ str           ┆ u32    ┆ f64         │
╞═══════════════╪════════╪═════════════╡
│ long (>10)    ┆ 279656 ┆ 0.028174    │
│ medium (4-10) ┆ 166502 ┆ 0.043317    │
│ short (<=3)   ┆ 192879 ┆ 0.064424    │
└───────────────┴────────┴─────────────┘

--- by query-category known/unknown ---
shape: (2, 3)
┌───────────────┬────────┬─────────────┐
│ unknown_query ┆ n      ┆ mean_ndcg10 │
│ ---           ┆ ---    ┆ ---         │
│ bool          ┆ u32    ┆ f64         │
╞═══════════════╪════════╪═════════════╡
│ false         ┆ 505829 ┆ 0.054401    │
│ true          ┆ 133208 ┆ 0.0         │
└───────────────┴────────┴─────────────┘


**Finding**: performance should be read alongside these slices, not just the aggregate - a model
that's strong on short/known-category sessions but weak on long/unknown-category ones has a real,
practically important limitation the aggregate number hides.

## Step 4: Calibration

Not applicable in the usual sense - LightGBM `lambdarank` scores are relative ranking scores, not
calibrated probabilities, and nothing downstream in this project consumes them as probabilities. Skipped
per the guide's own condition ("if predicted probabilities will be used directly").

## Step 5: Error analysis

In [4]:
no_hit = joined.filter(pl.col("ndcg10") == 0.0).height
print(f"examples with zero NDCG@10 (true item not recovered in top 10): {no_hit:,} / {joined.height:,} ({no_hit/joined.height:.1%})")

empty_pool = joined.filter(pl.col("gains").is_null()).height
print(f"of those, {empty_pool:,} had a completely empty candidate pool (Phase 4's retrieval ceiling, not a reranking failure)")
print(f"the remaining {no_hit - empty_pool:,} had candidates but the reranker didn't surface the true item in the top 10")

examples with zero NDCG@10 (true item not recovered in top 10): 597,397 / 639,037 (93.5%)
of those, 28,629 had a completely empty candidate pool (Phase 4's retrieval ceiling, not a reranking failure)
the remaining 568,768 had candidates but the reranker didn't surface the true item in the top 10


This separates two different failure modes that get conflated in the aggregate NDCG number: **retrieval
failures** (the co-visitation candidate pool never contained the true item - nothing the reranker could
have done) versus **ranking failures** (the item was retrievable but not ranked highly enough). Only
the second is something Phase 6's model choice can improve on; the first is a Phase 4/candidate-pool
limitation.

## Step 6: Compare test and validation results

In [5]:
print(f"val  reranker NDCG@10: 0.0577 [0.0571, 0.0584]")
print(f"test reranker NDCG@10: {r_mean:.4f} [{r_lo:.4f}, {r_hi:.4f}]")
gap = r_mean - 0.0577
print(f"gap: {gap:+.4f}")

val  reranker NDCG@10: 0.0577 [0.0571, 0.0584]
test reranker NDCG@10: 0.0431 [0.0426, 0.0435]
gap: -0.0146


A gap here is explained by the same point-in-time generalization effect found in Phase 4: test is
chronologically furthest from the training window, so its examples reference more items and patterns
the frozen co-visitation/category artifacts never saw. Any gap in this direction is expected, not a
red flag - it would be a red flag only if test *outperformed* val by a wide margin (which would suggest
a leak) or if the gap were far larger than Phase 4's retrieval-ceiling numbers would predict.

## Go / no-go decision

The reranker beats the co-visitation baseline on the locked test set by the same kind of margin seen on
val (paired bootstrap p-value above), so the decision is **go**: the LightGBM reranker is the Level 0
Phase 6 model, and its test-set numbers above are the honest, once-only estimate of how it performs.

## Exit criteria

Test metrics against baseline (Step 1), operating "threshold" decision (Step 2 - K=10 stands, no
probability threshold applies), slice table (Step 3), go/no-go decision (above). Met. The test set has
now been used exactly once, as this notebook's own opening cell commits to.